# 第 35 章：LLM 辅助科研编程、代码验证与 notebook 工作流

这个 notebook 不直接调用任何在线 LLM API；它演示的是另一件更重要的事：

- 当 AI 助手帮你写出一个“看起来合理”的科研函数时，怎样建立验证回路
- 怎样用一个极小的天文教学数据集，把 bug、假设、测试和 notebook workflow 串起来
- 怎样把 AI 助手从“替你回答”变成“帮你加速检查、重构和解释”的合作者

教学说明：本章继续保持 pure-Python、smoke-test-friendly 风格。重点不是某个具体平台，而是可复现、可验证、可审计的科研工作流。


In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path
from statistics import median

DATA_PATH = Path("../../data/small/lightcurve_debug_demo.csv").resolve()

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

for row in rows:
    row["time_days"] = float(row["time_days"])
    row["relative_flux"] = float(row["relative_flux"])
    row["flux_error"] = float(row["flux_error"])

print(f"Loaded {len(rows)} cadences from {DATA_PATH.name}")
print("segment counts:", dict(Counter(row["segment_label"] for row in rows)))
print("quality counts:", dict(Counter(row["quality_flag"] for row in rows)))
print("First four rows:")
for row in rows[:4]:
    print(row)


In [ ]:
def select_rows(items, segment=None, quality=None):
    selected = []
    for row in items:
        if segment is not None and row["segment_label"] != segment:
            continue
        if quality is not None and row["quality_flag"] != quality:
            continue
        selected.append(row)
    return selected


clean_rows = select_rows(rows, quality="clean")
clean_out_of_transit = select_rows(clean_rows, segment="out_of_transit")
clean_in_transit = select_rows(clean_rows, segment="in_transit")
flagged_rows = [row for row in rows if row["quality_flag"] != "clean"]

reference_depth = median(row["relative_flux"] for row in clean_out_of_transit) - median(
    row["relative_flux"] for row in clean_in_transit
)

print("Flagged rows that should not dominate the scientific result:")
for row in flagged_rows:
    print(row["cadence_id"], row["quality_flag"], row["relative_flux"])
print("clean out-of-transit median:", round(median(row["relative_flux"] for row in clean_out_of_transit), 4))
print("clean in-transit median:", round(median(row["relative_flux"] for row in clean_in_transit), 4))
print("reference transit depth from clean medians:", round(reference_depth, 4))


In [ ]:
AI_DRAFT_CODE = '''def ai_draft_transit_depth(lightcurve_rows):
    fluxes = [row["relative_flux"] for row in lightcurve_rows]
    return max(fluxes) - min(fluxes)
'''


def ai_draft_transit_depth(lightcurve_rows):
    fluxes = [row["relative_flux"] for row in lightcurve_rows]
    return max(fluxes) - min(fluxes)


draft_depth_all_rows = ai_draft_transit_depth(rows)
draft_depth_clean_rows = ai_draft_transit_depth(clean_rows)

print("AI draft code:")
print(AI_DRAFT_CODE)
print("draft depth using all rows:", round(draft_depth_all_rows, 4))
print("draft depth using only clean rows:", round(draft_depth_clean_rows, 4))
print("difference caused by flagged rows:", round(abs(draft_depth_all_rows - draft_depth_clean_rows), 4))


In [ ]:
def verification_report(depth_fn, lightcurve_rows):
    clean_only = [row for row in lightcurve_rows if row["quality_flag"] == "clean"]
    clean_out = [row for row in clean_only if row["segment_label"] == "out_of_transit"]
    clean_in = [row for row in clean_only if row["segment_label"] == "in_transit"]

    candidate_depth = depth_fn(lightcurve_rows)
    clean_depth = depth_fn(clean_only)
    baseline_median = median(row["relative_flux"] for row in clean_out)
    transit_median = median(row["relative_flux"] for row in clean_in)
    reference = baseline_median - transit_median

    checks = [
        ("physical_range", 0.0 < candidate_depth < 0.03, f"depth={candidate_depth:.4f}"),
        (
            "robust_to_flagged_rows",
            abs(candidate_depth - clean_depth) < 0.005,
            f"|all-clean|={abs(candidate_depth - clean_depth):.4f}",
        ),
        (
            "matches_reference_median",
            abs(candidate_depth - reference) < 0.005,
            f"|candidate-reference|={abs(candidate_depth - reference):.4f}",
        ),
        (
            "transit_dimmer_than_baseline",
            transit_median < baseline_median,
            f"baseline={baseline_median:.4f}, transit={transit_median:.4f}",
        ),
    ]

    return {
        "candidate_depth": candidate_depth,
        "clean_depth": clean_depth,
        "reference_depth": reference,
        "checks": [
            {"name": name, "passed": passed, "detail": detail}
            for name, passed, detail in checks
        ],
    }


def print_report(name, report):
    print(name)
    print("candidate depth:", round(report["candidate_depth"], 4))
    print("clean-only depth:", round(report["clean_depth"], 4))
    print("reference depth:", round(report["reference_depth"], 4))
    for item in report["checks"]:
        status = "PASS" if item["passed"] else "FAIL"
        print(f"  {status:4s} {item['name']}: {item['detail']}")


draft_report = verification_report(ai_draft_transit_depth, rows)
print_report("Verification report for the AI draft:", draft_report)


In [ ]:
def verified_transit_depth(lightcurve_rows):
    clean_only = [row for row in lightcurve_rows if row["quality_flag"] == "clean"]
    out_of_transit_fluxes = [row["relative_flux"] for row in clean_only if row["segment_label"] == "out_of_transit"]
    in_transit_fluxes = [row["relative_flux"] for row in clean_only if row["segment_label"] == "in_transit"]
    return median(out_of_transit_fluxes) - median(in_transit_fluxes)


verified_report = verification_report(verified_transit_depth, rows)
print_report("Verification report for the repaired estimator:", verified_report)
print("improvement over AI draft:", round(draft_report["candidate_depth"] - verified_report["candidate_depth"], 4))

if not all(item["passed"] for item in verified_report["checks"]):
    raise AssertionError("Verified estimator failed the regression checks.")


In [ ]:
assistant_review_prompt = f"""你是我的科研代码审查助手。

任务：
1. 审查下面这段 transit depth 函数是否会被坏点污染；
2. 列出潜在 bug、隐含假设和需要补上的测试；
3. 给出一个更稳妥的修正版，但不要跳过解释。

数据列：
- cadence_id
- time_days (day)
- relative_flux (normalized flux)
- flux_error
- segment_label (out_of_transit / in_transit)
- quality_flag (clean / cosmic_ray)

已知科学约束：
- transit depth 应该为正；
- 本教学数据里的合理 depth 大约在 0 到 0.03 之间；
- 删除 flagged cadence 后，结果不应该大幅漂移；
- clean out-of-transit median - clean in-transit median 可以作为参考计算。

待审查代码：
{AI_DRAFT_CODE}

请按下面格式输出：
- Potential bugs
- Hidden assumptions
- Suggested tests
- Safer implementation
"""

print(assistant_review_prompt)


In [ ]:
workflow_checklist = [
    "先把数据 schema、单位、坏点策略和科学约束明确写给 AI 助手。",
    "要求 AI 一起给出测试、失败模式和假设，而不是只给一段代码。",
    "把关键结果冻结成可重复执行的回归检查。",
    "重构后重新运行整份 notebook，而不是只看当前单元格。",
    "把最终保留的实现和验证结果一起写进报告或脚注。",
]

print("Notebook workflow checklist:")
for index, item in enumerate(workflow_checklist, start=1):
    print(f"{index}. {item}")

regression_snapshot = {
    "python_version": platform.python_version(),
    "data_file": DATA_PATH.name,
    "cadence_count": len(rows),
    "flagged_count": len(flagged_rows),
    "reference_depth": round(reference_depth, 4),
    "verified_depth": round(verified_report["candidate_depth"], 4),
    "all_checks_pass": all(item["passed"] for item in verified_report["checks"]),
}

print("Regression snapshot:")
for key, value in regression_snapshot.items():
    print(f"  {key}: {value}")


In [ ]:
try:
    import matplotlib.pyplot as plt

    times = [row["time_days"] for row in rows]
    fluxes = [row["relative_flux"] for row in rows]
    colors = ["#c0392b" if row["quality_flag"] != "clean" else "#2f6b99" for row in rows]

    plt.figure(figsize=(7.4, 3.6))
    plt.scatter(times, fluxes, c=colors, s=45)
    plt.axhline(median(row["relative_flux"] for row in clean_out_of_transit), color="#1f7a4d", linestyle="--", label="clean baseline median")
    plt.axhline(median(row["relative_flux"] for row in clean_in_transit), color="#8e5ea2", linestyle=":", label="clean transit median")
    plt.xlabel("time [day]")
    plt.ylabel("normalized flux")
    plt.title("Tiny light-curve debugging dataset")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f"Optional matplotlib plot skipped: {exc}")


In [ ]:
from part5_delivery_exports import export_ch35_llm_programming_delivery

export_ch35_llm_programming_delivery(globals())
